# CrackSpot — Notebook điều khiển chính

Đây là entrypoint chính để điều khiển toàn bộ đồ án. Notebook **không chép lại business logic**: mọi bước đều gọi package `crackspot` hoặc CLI trong `scripts/`.

Luồng chuẩn: **preflight → data gate → smoke → E1/E2/E3 → chọn E2/E3 → E4 → khóa E5 → final test one-shot → ảnh tự chụp/benchmark/Grad-CAM → report assets → demo**.

> Không bật final test khi chưa khóa model/threshold. Smoke luôn là `NOT_VALID_FOR_REPORT`. Hiện repository chưa có Git HEAD nên official training sẽ chủ động bị chặn cho đến khi người dùng tạo commit đầu tiên.

## 0. Cách sử dụng

1. Local: chạy các cell từ trên xuống. Colab: clone repository bằng các tham số `COLAB_*`, sau đó bật `INSTALL_DEPENDENCIES` và `DO_DOWNLOAD_SDNET2018`.
2. Chỉ sửa cell **THAM SỐ ĐIỀU KHIỂN**.
3. Mỗi lần chỉ bật các cờ `DO_*` cần chạy rồi chạy cell tương ứng.
4. Không đổi `RUN_IDS` giữa lúc resume. Không xóa/recreate `split_v1` sau khi xem test.
5. Final test yêu cầu cả danh sách target và chuỗi xác nhận chính xác.

In [11]:
# ==================== THAM SỐ ĐIỀU KHIỂN ====================
PROJECT_ROOT_OVERRIDE = ""  # Ví dụ Colab: /content/CrackSpot
CLONE_REPOSITORY_IN_COLAB = True  # Tự clone vào /content/CrackSpot khi Colab chưa có source
COLAB_REPOSITORY_URL = "https://github.com/MKPPF/Do_An_Thi_Giac_May_Tinh.git"
COLAB_REPOSITORY_REF = "main"

INSTALL_DEPENDENCIES = None  # None = tự cài trên Colab, không tự cài ở local
RUN_QUALITY_GATE = False
SHOW_PROJECT_INVENTORY = True
SHOW_SPLIT_PREVIEW = True
DO_DOWNLOAD_SDNET2018 = False  # Colab cần bật nếu chưa có data/raw/SDNET2018
VERIFY_ALL_IMAGE_BYTES = False  # Quét đủ 56.088 ảnh; official training cũng tự quét
RUN_SMOKE = False

DO_TRAIN_E1 = False
DO_TRAIN_E2 = False
DO_TRAIN_E3 = False
DO_SELECT_E2_E3 = False
DO_TRAIN_E4 = False
RESUME_TRAINING = False
DO_LOCK_FIXED_SELECTIONS = False  # Khóa E1-E3 ở threshold 0.5
DO_TUNE_AND_LOCK_E5 = False
DO_EXPORT_SELECTED_METADATA = False

# Chỉ điền target cần mở test, ví dụ ["E1"] hoặc ["E1", "E2", "E3", "E5"].
FINAL_TEST_TARGETS = []
FINAL_TEST_CONFIRMATION = ""
REQUIRED_FINAL_CONFIRMATION = "I_UNDERSTAND_FINAL_TEST_IS_ONE_SHOT"

DO_AUGMENTATION_FIGURE = False
DO_GRADCAM_GRID = False
DO_BENCHMARK = False
DO_REAL_IMAGE_EVALUATION = False
DO_GENERATE_REPORT = False

RUN_IDS = {
    "E1": "e1-official-v1",
    "E2": "e2-official-v1",
    "E3": "e3-official-v1",
    "E4": "e4-official-v1",
}

# Bắt buộc thay bằng ảnh thật khi benchmark/đánh giá ngoài miền.
BENCHMARK_IMAGE = "data/external/real/probe.jpg"
REAL_MANIFEST = "data/external/real/manifest.csv"
# =============================================================

In [12]:
import importlib
import json
import os
import platform
import shlex
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
override_path = (
    Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve() if PROJECT_ROOT_OVERRIDE else None
)
candidates = [
    path
    for path in (override_path, Path.cwd(), Path.cwd() / "CrackSpot", Path("/content/CrackSpot"))
    if path is not None
]
PROJECT_ROOT = next(
    (path.resolve() for path in candidates if (path / "pyproject.toml").is_file()),
    None,
)

if PROJECT_ROOT is None and IN_COLAB and CLONE_REPOSITORY_IN_COLAB:
    if not COLAB_REPOSITORY_URL.strip():
        raise ValueError("COLAB_REPOSITORY_URL đang rỗng; không thể clone source.")
    clone_target = override_path or Path("/content/CrackSpot")
    if clone_target.exists():
        raise FileExistsError(
            f"{clone_target} đã tồn tại nhưng thiếu pyproject.toml. "
            "Chọn Runtime > Disconnect and delete runtime rồi chạy lại."
        )
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            COLAB_REPOSITORY_REF,
            "--single-branch",
            COLAB_REPOSITORY_URL,
            str(clone_target),
        ],
        check=True,
    )
    PROJECT_ROOT = clone_target.resolve()

if PROJECT_ROOT is None:
    searched = ", ".join(str(path) for path in candidates)
    raise FileNotFoundError(
        f"Không tìm thấy pyproject.toml. Đã kiểm tra: {searched}. "
        "Trên Colab hãy để CLONE_REPOSITORY_IN_COLAB=True."
    )
os.chdir(PROJECT_ROOT)
source_root = str(PROJECT_ROOT / "src")
if source_root not in sys.path:
    sys.path.insert(0, source_root)


def run(command: list[str], *, enabled: bool = True, description: str = "") -> None:
    if not enabled:
        print(f"SKIP: {description or command[0]}")
        return
    printable = (
        subprocess.list2cmdline([str(part) for part in command])
        if os.name == "nt"
        else shlex.join([str(part) for part in command])
    )
    print(f"\n>>> {description}\n{printable}")
    subprocess.run([str(part) for part in command], cwd=PROJECT_ROOT, check=True)


PYTHON = sys.executable
DATA_ARCHIVE = PROJECT_ROOT / "data/raw/SDNET2018.zip"
DATASET_ROOT = PROJECT_ROOT / "data/raw/SDNET2018"
SPLIT_DIR = PROJECT_ROOT / "data/manifests/split_v1"
LOCKED_MANIFEST = SPLIT_DIR / "manifest.csv"
RUN_ROOT = PROJECT_ROOT / "artifacts/runs"
REPORT_ROOT = PROJECT_ROOT / "artifacts/report"
REPORT_BUNDLE = REPORT_ROOT / "final_bundle_v1"
MODEL_SELECTION = PROJECT_ROOT / "artifacts/model_selection.json"
RUN_DIRS = {key: RUN_ROOT / value for key, value in RUN_IDS.items()}

print("Project root :", PROJECT_ROOT)
print("Python       :", sys.version.split()[0])
print("Platform     :", platform.platform())
print("Colab        :", IN_COLAB)
print("Manifest     :", LOCKED_MANIFEST)
git_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
).stdout.strip()
print("Dataset root :", DATASET_ROOT)
print("Git commit   :", git_commit or "MISSING")

Project root : /content/CrackSpot
Python       : 3.13.15
Platform     : Linux-6.6.122+-x86_64-with-glibc2.35
Colab        : True
Manifest     : /content/CrackSpot/data/manifests/split_v1/manifest.csv
Dataset root : /content/CrackSpot/data/raw/SDNET2018
Git commit   : 98207c380e179ff4764691c63b755b33fe63af8b


## 1. Cài dependency và kiểm tra thiết bị

In [13]:
SHOULD_INSTALL_DEPENDENCIES = (
    IN_COLAB if INSTALL_DEPENDENCIES is None else bool(INSTALL_DEPENDENCIES)
)
if SHOULD_INSTALL_DEPENDENCIES:
    run(
        [PYTHON, "-m", "pip", "install", "-r", "requirements-dev.txt"],
        description="Cài dependency train/demo/test",
    )
    run([PYTHON, "-m", "pip", "install", "-e", "."], description="Cài package editable")

tf = importlib.import_module("tensorflow")

print("TensorFlow:", tf.__version__)
print("Devices:", tf.config.list_physical_devices())
run(
    ["nvidia-smi"],
    enabled=bool(tf.config.list_physical_devices("GPU")),
    description="Thông tin GPU",
)


>>> Cài dependency train/demo/test
/usr/bin/python3 -m pip install -r requirements-dev.txt


CalledProcessError: Command '['/usr/bin/python3', '-m', 'pip', 'install', '-r', 'requirements-dev.txt']' returned non-zero exit status 1.

## 2. Quality gate và Git preflight

In [ ]:
run(
    [PYTHON, "-m", "ruff", "format", "--check", "."],
    enabled=RUN_QUALITY_GATE,
    description="Ruff format check",
)
run([PYTHON, "-m", "ruff", "check", "."], enabled=RUN_QUALITY_GATE, description="Ruff lint")
run([PYTHON, "-m", "pytest", "-q"], enabled=RUN_QUALITY_GATE, description="Full test suite")
run([PYTHON, "-m", "pip", "check"], enabled=RUN_QUALITY_GATE, description="Dependency check")

git_head = subprocess.run(
    ["git", "rev-parse", "--verify", "HEAD"], cwd=PROJECT_ROOT, capture_output=True, text=True
)
git_status = subprocess.run(
    ["git", "status", "--porcelain=v1", "--untracked-files=normal"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(
    "Git HEAD:",
    git_head.stdout.strip()
    if git_head.returncode == 0
    else "MISSING — official training đang bị khóa",
)
print("Git worktree:", "CLEAN" if not git_status.stdout.strip() else git_status.stdout)

## 3. Dữ liệu Colab, inventory, locked split và bytes ảnh

Không sao chép 56.088 ảnh thành ba thư mục. `train.csv`, `validation.csv` và `test.csv` là ba tập chính thức; ảnh gốc giữ nguyên dưới `data/raw/SDNET2018`.

```text
data/manifests/
├── audit_manifest.csv
├── data_audit.json
├── pre_split_curation_v1/
│   ├── conflict_report.json
│   ├── conflict_rows.csv
│   └── pre_split_manifest.csv
└── split_v1/
    ├── train.csv          # 39.014 ảnh / 160 source groups
    ├── validation.csv     # 8.540 ảnh / 35 source groups
    ├── test.csv           # 8.534 ảnh / 35 source groups
    ├── manifest.csv       # gộp ba split, canonical locked manifest
    ├── split_audit.json   # zero path/group/hash overlap
    ├── manifest_hashes.json
    ├── split_complete.json
    └── snapshot lineage của audit/curation
```

Các manifest được Git mang theo lên Colab; ZIP và ảnh gốc được tải riêng bằng `DO_DOWNLOAD_SDNET2018`.

In [ ]:
import pandas as pd

from crackspot.data import (
    load_manifest_table,
    verify_locked_split_bundle,
    verify_official_dataset_preconditions,
)

if DO_DOWNLOAD_SDNET2018 and not DATASET_ROOT.is_dir():
    run(
        [
            PYTHON,
            "scripts/download_sdnet2018.py",
            "--archive",
            str(DATA_ARCHIVE),
            "--extract-dir",
            str(DATA_ARCHIVE.parent),
        ],
        description="Tải, xác minh MD5 và giải nén SDNET2018",
    )
elif DO_DOWNLOAD_SDNET2018:
    print("DATASET READY: bỏ qua download vì data/raw/SDNET2018 đã tồn tại.")
elif not DATASET_ROOT.is_dir():
    print("DATASET MISSING: Colab hãy bật DO_DOWNLOAD_SDNET2018 trước khi train.")


def file_size_text(size: int) -> str:
    value = float(size)
    for unit in ("B", "KB", "MB", "GB"):
        if value < 1024 or unit == "GB":
            return f"{value:.1f} {unit}"
        value /= 1024
    raise AssertionError("unreachable")


def git_file_state(path: Path) -> str:
    relative = path.resolve().relative_to(PROJECT_ROOT).as_posix()
    tracked = (
        subprocess.run(
            ["git", "ls-files", "--error-unmatch", relative], cwd=PROJECT_ROOT, capture_output=True
        ).returncode
        == 0
    )
    if tracked:
        return "TRACKED"
    ignored = (
        subprocess.run(["git", "check-ignore", "-q", relative], cwd=PROJECT_ROOT).returncode == 0
    )
    return "IGNORED" if ignored else "UNTRACKED — cần git add/commit"


if SHOW_PROJECT_INVENTORY:
    print("\n=== TOÀN BỘ TỆP DỰ ÁN (bỏ qua .git/.venv/cache/ảnh runtime) ===")
    skipped_names = {
        ".git",
        ".venv",
        "__pycache__",
        ".pytest_cache",
        ".ruff_cache",
        ".ipynb_checkpoints",
    }
    skipped_runtime = {"data/raw", "artifacts/runs", "artifacts/smoke", "artifacts/report"}
    inventory = []
    for current, directories, filenames in os.walk(PROJECT_ROOT):
        current_path = Path(current)
        relative_dir = current_path.relative_to(PROJECT_ROOT).as_posix()
        if any(
            relative_dir == root or relative_dir.startswith(root + "/") for root in skipped_runtime
        ):
            directories[:] = []
            continue
        directories[:] = sorted(
            name
            for name in directories
            if name not in skipped_names and not name.endswith(".egg-info")
        )
        inventory.extend(
            (current_path / name).relative_to(PROJECT_ROOT).as_posix() for name in sorted(filenames)
        )
    print("\n".join(inventory))

    manifest_rows = []
    for path in sorted((PROJECT_ROOT / "data/manifests").rglob("*")):
        if path.is_file():
            manifest_rows.append(
                {
                    "file": path.relative_to(PROJECT_ROOT).as_posix(),
                    "size": file_size_text(path.stat().st_size),
                    "git": git_file_state(path),
                }
            )
    print("\n=== TOÀN BỘ MANIFEST/AUDIT/SPLIT ===")
    print(pd.DataFrame(manifest_rows).to_string(index=False))

    print("\n=== DỮ LIỆU ẢNH GỐC (tóm tắt, không in 56.088 tên ảnh) ===")
    raw_image_count = 0
    for relative in ("D/CD", "D/UD", "P/CP", "P/UP", "W/CW", "W/UW"):
        folder = DATASET_ROOT / relative
        count = sum(1 for path in folder.iterdir() if path.is_file()) if folder.is_dir() else 0
        raw_image_count += count
        print(f"{relative:5} {count:>6} ảnh  {'READY' if folder.is_dir() else 'MISSING'}")
    if raw_image_count:
        print(
            f"RAW TOTAL: {raw_image_count:,}; locked split: 56.088 (loại 4 dòng/2 hash mâu thuẫn trước split)."
        )

bundle = verify_locked_split_bundle(SPLIT_DIR)
print("Status:", bundle.completion["status"])
print("Canonical manifest SHA-256:", bundle.manifest_sha256)
print("Counts:", {name: row["images"] for name, row in bundle.audit["counts"].items()})
print("Leakage audit valid:", bundle.audit["valid"])

if SHOW_SPLIT_PREVIEW:
    for split_name in ("train", "validation", "test"):
        split_path = SPLIT_DIR / f"{split_name}.csv"
        split_frame = pd.read_csv(split_path)
        print(
            f"\n=== {split_name.upper()} | {len(split_frame):,} ảnh | {split_frame['source_group'].nunique():,} nhóm ==="
        )
        print(split_frame.groupby(["surface", "label"]).size().rename("images").to_string())
        print("\n3 dòng đầu:")
        print(
            split_frame[["relative_path", "label", "surface", "source_group", "sha256"]]
            .head(3)
            .to_string(index=False)
        )

if VERIFY_ALL_IMAGE_BYTES:
    frame = load_manifest_table(LOCKED_MANIFEST)
    evidence = verify_official_dataset_preconditions(frame, LOCKED_MANIFEST, DATASET_ROOT)
    print(json.dumps(evidence.to_dict(), ensure_ascii=False, indent=2))

## 4. Smoke pipeline kỹ thuật

In [ ]:
run(
    [PYTHON, "scripts/smoke_pipeline.py"],
    enabled=RUN_SMOKE,
    description="Smoke train → threshold → final smoke → CLI → Grad-CAM",
)

## 5. Huấn luyện E1–E3

Official run yêu cầu Git HEAD hợp lệ, tracked worktree sạch, locked split và full byte-integrity pass.

In [ ]:
CONFIGS = {
    "E1": "configs/experiments/e1_baseline.yaml",
    "E2": "configs/experiments/e2_finetune_basic.yaml",
    "E3": "configs/experiments/e3_finetune_deep.yaml",
    "E4": "configs/experiments/e4_augmentation.yaml",
}

for experiment, enabled in (("E1", DO_TRAIN_E1), ("E2", DO_TRAIN_E2), ("E3", DO_TRAIN_E3)):
    command = [
        PYTHON,
        "scripts/run_experiment.py",
        "--config",
        CONFIGS[experiment],
        "--manifest",
        str(LOCKED_MANIFEST),
        "--dataset-root",
        str(DATASET_ROOT),
        "--output-root",
        str(RUN_ROOT),
        "--run-id",
        RUN_IDS[experiment],
    ]
    if RESUME_TRAINING:
        command.append("--resume")
    run(command, enabled=enabled, description=f"Train {experiment}")

## 6. Chọn E2/E3 và huấn luyện E4

In [ ]:
run(
    [
        PYTHON,
        "scripts/select_model.py",
        "--e2-run",
        str(RUN_DIRS["E2"]),
        "--e3-run",
        str(RUN_DIRS["E3"]),
        "--output",
        str(MODEL_SELECTION),
        "--project-root",
        str(PROJECT_ROOT),
    ],
    enabled=DO_SELECT_E2_E3,
    description="Chọn E2/E3 chỉ bằng validation val_loss",
)

e4_command = [
    PYTHON,
    "scripts/run_experiment.py",
    "--config",
    CONFIGS["E4"],
    "--manifest",
    str(LOCKED_MANIFEST),
    "--dataset-root",
    str(DATASET_ROOT),
    "--output-root",
    str(RUN_ROOT),
    "--run-id",
    RUN_IDS["E4"],
    "--model-selection",
    str(MODEL_SELECTION),
]
if RESUME_TRAINING:
    e4_command.append("--resume")
run(e4_command, enabled=DO_TRAIN_E4, description="Train E4 với augmentation và selection đã khóa")

## 7. Khóa selection E1–E3 và tune/khóa E5

In [ ]:
if DO_LOCK_FIXED_SELECTIONS:
    for experiment in ("E1", "E2", "E3"):
        run(
            [
                PYTHON,
                "scripts/lock_selection.py",
                "--run-dir",
                str(RUN_DIRS[experiment]),
                "--experiment",
                experiment,
                "--threshold",
                "0.5",
            ],
            description=f"Khóa {experiment} threshold 0.5",
        )
else:
    print("SKIP: lock E1-E3")

E4_RUN = RUN_DIRS["E4"]
THRESHOLD_RESULT = E4_RUN / "threshold_validation.json"
SELECTED_METADATA = E4_RUN / "selected_model.metadata.json"
if DO_TUNE_AND_LOCK_E5:
    run(
        [
            PYTHON,
            "-m",
            "crackspot.modeling.threshold",
            "--predictions",
            str(E4_RUN / "predictions_validation.csv"),
            "--output",
            str(THRESHOLD_RESULT),
        ],
        description="Tune E5 threshold trên validation",
    )
    run(
        [
            PYTHON,
            "scripts/lock_selection.py",
            "--run-dir",
            str(E4_RUN),
            "--experiment",
            "E5",
            "--threshold-result",
            str(THRESHOLD_RESULT),
        ],
        description="Khóa E5 selection",
    )
else:
    print("SKIP: tune/lock E5")

run(
    [
        PYTHON,
        "scripts/export_selected_metadata.py",
        "--selection",
        str(E4_RUN / "selection_complete.json"),
        "--output",
        str(SELECTED_METADATA),
    ],
    enabled=DO_EXPORT_SELECTED_METADATA,
    description="Xuất metadata demo với threshold E5 đã khóa",
)

## 8. Final test — one-shot

Mỗi checkpoint chỉ được cấp quyền final-test một lần. Cell từ chối chạy nếu thiếu chuỗi xác nhận hoặc target ngoài `E1/E2/E3/E5`.

In [ ]:
allowed_targets = {"E1", "E2", "E3", "E5"}
requested_targets = [str(item).upper() for item in FINAL_TEST_TARGETS]
if requested_targets:
    if FINAL_TEST_CONFIRMATION != REQUIRED_FINAL_CONFIRMATION:
        raise RuntimeError("Sai/thiếu FINAL_TEST_CONFIRMATION; test vẫn bị khóa.")
    if len(requested_targets) != len(set(requested_targets)) or not set(requested_targets).issubset(
        allowed_targets
    ):
        raise ValueError("FINAL_TEST_TARGETS chỉ nhận mỗi target một lần trong E1/E2/E3/E5.")
    for experiment in requested_targets:
        run_dir = E4_RUN if experiment == "E5" else RUN_DIRS[experiment]
        output = REPORT_ROOT / "final_evaluation" / f"{experiment.lower()}-{run_dir.name}"
        run(
            [
                PYTHON,
                "scripts/evaluate_final.py",
                "--selection",
                str(run_dir / "selection_complete.json"),
                "--manifest",
                str(LOCKED_MANIFEST),
                "--dataset-root",
                str(DATASET_ROOT),
                "--output-dir",
                str(output),
            ],
            description=f"FINAL TEST {experiment} — ONE SHOT",
        )
else:
    print("FINAL TEST LOCKED: FINAL_TEST_TARGETS đang rỗng.")

## 9. Hình augmentation, Grad-CAM, benchmark và ảnh tự chụp

In [ ]:
E5_EVAL = REPORT_ROOT / "final_evaluation" / f"e5-{E4_RUN.name}"
AUGMENTATION_PNG = REPORT_ROOT / "fig_augmentation_before_after.png"
GRADCAM_PNG = REPORT_ROOT / "fig_gradcam_tp_tn_fp_fn.png"
BENCHMARK_JSON = PROJECT_ROOT / "artifacts/benchmarks" / f"{E4_RUN.name}.json"
REAL_EVAL = REPORT_ROOT / "real_images" / E4_RUN.name

run(
    [
        PYTHON,
        "scripts/visualize_augmentation.py",
        "--config",
        str(E4_RUN / "config_snapshot.json"),
        "--run-dir",
        str(E4_RUN),
        "--manifest",
        str(LOCKED_MANIFEST),
        "--dataset-root",
        str(DATASET_ROOT),
        "--output",
        str(AUGMENTATION_PNG),
    ],
    enabled=DO_AUGMENTATION_FIGURE,
    description="Sinh hình augmentation report-valid",
)
run(
    [
        PYTHON,
        "scripts/generate_gradcam_grid.py",
        "--selection",
        str(E4_RUN / "selection_complete.json"),
        "--predictions",
        str(E5_EVAL / "predictions_test.csv"),
        "--dataset-root",
        str(DATASET_ROOT),
        "--output",
        str(GRADCAM_PNG),
    ],
    enabled=DO_GRADCAM_GRID,
    description="Sinh grid Grad-CAM TP/TN/FP/FN",
)
run(
    [
        PYTHON,
        "scripts/benchmark_inference.py",
        str(PROJECT_ROOT / BENCHMARK_IMAGE),
        "--model",
        str(E4_RUN / "model.keras"),
        "--metadata",
        str(SELECTED_METADATA),
        "--output",
        str(BENCHMARK_JSON),
        "--warmup-runs",
        "10",
        "--measured-runs",
        "100",
    ],
    enabled=DO_BENCHMARK,
    description="Benchmark latency E5 checkpoint",
)
run(
    [
        PYTHON,
        "scripts/evaluate_real_images.py",
        "--selection",
        str(E4_RUN / "selection_complete.json"),
        "--manifest",
        str(PROJECT_ROOT / REAL_MANIFEST),
        "--dataset-root",
        str((PROJECT_ROOT / REAL_MANIFEST).parent),
        "--output-dir",
        str(REAL_EVAL),
        "--confirm-self-captured",
    ],
    enabled=DO_REAL_IMAGE_EVALUATION,
    description="Đánh giá ảnh nhóm tự chụp tách riêng",
)

## 10. Tổng hợp report facts chính thức

In [ ]:
if DO_GENERATE_REPORT:
    report_command = [PYTHON, "scripts/generate_report_assets.py"]
    for experiment in ("E1", "E2", "E3"):
        report_command += [
            "--evaluation-dir",
            str(
                REPORT_ROOT
                / "final_evaluation"
                / f"{experiment.lower()}-{RUN_DIRS[experiment].name}"
            ),
        ]
    report_command += ["--evaluation-dir", str(E5_EVAL)]
    report_command += ["--manifest", str(LOCKED_MANIFEST), "--split-bundle-dir", str(SPLIT_DIR)]
    report_command += [
        "--validation-predictions",
        str(E4_RUN / "predictions_validation.csv"),
        "--threshold-result",
        str(THRESHOLD_RESULT),
    ]
    for experiment in ("E1", "E2", "E3", "E4"):
        report_command += ["--training-run-dir", str(RUN_DIRS[experiment])]
    report_command += ["--benchmark", str(BENCHMARK_JSON), "--real-evaluation-dir", str(REAL_EVAL)]
    report_command += [
        "--augmentation-sidecar",
        str(AUGMENTATION_PNG.with_suffix(".json")),
        "--gradcam-sidecar",
        str(GRADCAM_PNG.with_suffix(".json")),
    ]
    report_command += ["--output-dir", str(REPORT_BUNDLE), "--project-root", str(PROJECT_ROOT)]
    run(report_command, description="Sinh report_facts và report assets")
else:
    print("SKIP: generate report")

## 11. CLI và Streamlit demo

In [ ]:
# Sau khi E5 đã khóa, chạy dự đoán một ảnh:
# run([PYTHON, "scripts/predict.py", BENCHMARK_IMAGE, "--model", str(E4_RUN / "model.keras"), "--metadata", str(SELECTED_METADATA)], description="CLI prediction")

# Chạy Streamlit trong terminal/notebook cell riêng:
# os.environ["CRACKSPOT_MODEL_PATH"] = str(E4_RUN / "model.keras")
# os.environ["CRACKSPOT_METADATA_PATH"] = str(SELECTED_METADATA)
# run([PYTHON, "-m", "streamlit", "run", "app.py"], description="Streamlit demo")

## 12. Điều kiện kết luận

Chỉ dùng số liệu trong `artifacts/report/final_bundle_v1/report_facts.json`. Không tuyên bố Accuracy ≥ 0,92 nếu final artifact không chứng minh. Không gọi Grad-CAM là segmentation. Không gọi smoke là kết quả đồ án. Hồ sơ vẫn **CHƯA SẴN SÀNG NỘP** nếu thiếu ảnh tự chụp, E1–E5 thật, report 35–45 trang hoặc slide.